In [106]:
import pandas as pd

df = pd.read_excel("data/train.xlsx")

df

,review_description,rating
0,شركه زباله و سواقين بتبرشم و مفيش حتي رقم للشك...,-1
1,خدمة الدفع عن طريق الكي نت توقفت عندي اصبح فقط...,1
2,تطبيق غبي و جاري حذفه ، عاملين اكواد خصم و لما...,-1
3,فعلا تطبيق ممتاز بس لو فى امكانية يتيح لمستخدم...,1
4,سيء جدا ، اسعار رسوم التوصيل لا تمت للواقع ب ص...,-1
...,...,...
32031,التطبيق اصبح سيء للغايه نقوم بطلب لا يتم وصول ...,-1
32032,y love you,1
32033,الباقه بتخلص وبشحن مرتين باقه اضافيه ١٠٠ جنيه,-1
32034,تطبيق فاشل وصلني الطلب ناقص ومش ينفع اعمل حاجة...,-1


In [107]:
import pandas as pd
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download("stopwords")
nltk.download("punkt")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mereh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mereh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [108]:
df = df.dropna()

df = df.drop_duplicates()

df["review_description"] = df["review_description"].astype(str)

In [109]:
arabic_stopwords = set(stopwords.words("arabic"))

negation_words = {
    "لا", "لم", "لن", "ليس", "ما", "غير",
    "بدون", "دون", "ولا", "ليسوا", "ليست"
}

arabic_stopwords = arabic_stopwords - negation_words

In [110]:
!pip install emoji


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [111]:
import re
import emoji
from nltk.tokenize import word_tokenize

# Function to preprocess Arabic text
def preprocess(text):

    # ==========================
    # Remove URLs
    # ==========================
    text = re.sub(r"http\S+|www\S+", "", text)

    # ==========================
    # Remove HTML Tags
    # ==========================
    text = re.sub(r"<.*?>", "", text)

    # ==========================
    # Remove Emojis
    # ==========================
    text = emoji.replace_emoji(text, replace="")

    # ==========================
    # Remove Mentions (@username)
    # and Hashtags (#topic)
    # ==========================
    text = re.sub(r"[@#]\w+", "", text)

    # ==========================
    # Remove English Letters
    # ==========================
    text = re.sub(r"[A-Za-z]+", "", text)

    # ==========================
    # Remove Numbers
    # (Arabic & English)
    # ==========================
    text = re.sub(r"[0-9٠-٩]", "", text)

    # ==========================
    # Remove Punctuation
    # ==========================
    text = re.sub(r"[^\w\s]", " ", text)

    # ==========================
    # Remove Arabic Diacritics
    # ==========================
    text = re.sub(r"[ًٌٍَُِّْـ]", "", text)

    # ==========================
    # Normalize Arabic Letters
    # ==========================
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)

    # ==========================
    # Remove Repeated Characters
    # Example:
    # راااائع -> رائع
    # حلوووو -> حلو
    # ==========================
    text = re.sub(r"(.)\1{2,}", r"\1", text)

    # ==========================
    # Keep Arabic Letters Only
    # ==========================
    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)

    # ==========================
    # Remove Extra Spaces
    # ==========================
    text = re.sub(r"\s+", " ", text).strip()

    # ==========================
    # Tokenization
    # ==========================
    words = word_tokenize(text)

    # ==========================
    # Remove Arabic Stopwords
    # ==========================
    words = [word for word in words if word not in arabic_stopwords]


    # ==========================
    # Join Tokens Again
    # ==========================
    return " ".join(words)

In [112]:
df["clean_text"] = df["review_description"].apply(preprocess)

df = df[df["clean_text"].str.strip() != ""]

df = df[df["clean_text"].str.split().str.len() >= 2]

df.reset_index(drop=True, inplace=True)

In [113]:

print(df["rating"].value_counts())

rating
 1    15091
-1    10355
 0     1360
Name: count, dtype: int64


In [114]:
df[["review_description", "clean_text", "rating"]].head(10)

,review_description,clean_text,rating
0,شركه زباله و سواقين بتبرشم و مفيش حتي رقم للشك...,شركه زباله سواقين بتبرشم مفيش حتي رقم للشكاوي ...,-1
1,خدمة الدفع عن طريق الكي نت توقفت عندي اصبح فقط...,خدمة الدفع طريق الكي نت توقفت عندي اصبح فقط ال...,1
2,تطبيق غبي و جاري حذفه ، عاملين اكواد خصم و لما...,تطبيق غبي جاري حذفه عاملين اكواد خصم نستخدمها ...,-1
3,فعلا تطبيق ممتاز بس لو فى امكانية يتيح لمستخدم...,فعلا تطبيق ممتاز امكانية يتيح لمستخدم التطبيق ...,1
4,سيء جدا ، اسعار رسوم التوصيل لا تمت للواقع ب ص...,سيء جدا اسعار رسوم التوصيل لا تمت للواقع صله,-1
5,قعد عشرين سنة يدور على سائق بس اما عن توصيل ال...,قعد سنة يدور علي سايق اما توصيل الاشياء جيد جدا,0
6,احلئ تطبيق,احلي تطبيق,1
7,رائع واو مدهش,رايع مدهش,1
8,مکو بس البحرین وعمان وغیرهه بس العراق مکو یعنی...,مکو البحرین وعمان وغیرهه العراق مکو یعنی نجمه ...,-1
9,تطبيق جميل يستاهل الخمس نجوم👍👍👍,تطبيق جميل يستاهل الخمس نجوم,1


In [115]:
lengths = df["clean_text"].str.split().apply(len)

print(lengths.describe())

count    26806.000000
mean         9.215922
std         10.854824
min          2.000000
25%          3.000000
50%          6.000000
75%         11.000000
max        321.000000
Name: clean_text, dtype: float64


In [116]:
from sklearn.model_selection import train_test_split

X = df["clean_text"]
y = df["rating"]

y = df["rating"].replace({
    -1: 0,
     0: 1,
     1: 2
})
x_train, x_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [117]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(zip(np.unique(y_train), class_weights))

print(class_weights)

{np.int64(0): np.float64(0.8628681796233704), np.int64(1): np.float64(6.569852941176471), np.int64(2): np.float64(0.5921139827700463)}


In [118]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

VOCAB_SIZE = 30000
MAX_LEN = 50

tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(x_train)

x_train = tokenizer.texts_to_sequences(x_train)
x_test = tokenizer.texts_to_sequences(x_test)

x_train = pad_sequences(
    x_train,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

x_test = pad_sequences(
    x_test,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

In [ ]:
import time
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

lstm = Sequential()

# Embedding Layer
lstm.add(
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128,
        input_length=MAX_LEN
    )
)

# LSTM Layer
lstm.add(
    LSTM(
        64,
        dropout=0.3,
        recurrent_dropout=0.3
    )
)

# Hidden Layer
lstm.add(Dense(64, activation="relu"))

# Dropout Layer
lstm.add(Dropout(0.5))

# Output Layer
lstm.add(Dense(3, activation="softmax"))

optimizer = Adam(learning_rate=0.0005)

lstm.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

start_time = time.time()

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = lstm.fit(
    x_train,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=64,
    callbacks=[early_stop],
    class_weight=class_weights
)

end_time = time.time()

training_time = end_time - start_time

print(f"Training Time: {training_time:.2f} seconds")
print(f"Training Time: {training_time/60:.2f} minutes")

c:\Users\mereh\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/20
269/269 ━━━━━━━━━━━━━━━━━━━━ 46s 134ms/step - accuracy: 0.3121 - loss: 1.1042 - val_accuracy: 0.0595 - val_loss: 1.1101
Epoch 2/20
269/269 ━━━━━━━━━━━━━━━━━━━━ 28s 85ms/step - accuracy: 0.2546 - loss: 1.1009 - val_accuracy: 0.5570 - val_loss: 1.0954
Epoch 3/20
269/269 ━━━━━━━━━━━━━━━━━━━━ 20s 75ms/step - accuracy: 0.4237 - loss: 1.0893 - val_accuracy: 0.6020 - val_loss: 1.0520
Epoch 4/20
269/269 ━━━━━━━━━━━━━━━━━━━━ 26s 95ms/step - accuracy: 0.6488 - loss: 1.0393 - val_accuracy: 0.7587 - val_loss: 0.9433
Epoch 5/20
269/269 ━━━━━━━━━━━━━━━━━━━━ 20s 75ms/step - accuracy: 0.7535 - loss: 0.9881 - val_accuracy: 0.7671 - val_loss: 0.9444
Epoch 6/20
269/269 ━━━━━━━━━━━━━━━━━━━━ 19s 70ms/step - accuracy: 0.7726 - loss: 0.9690 - val_accuracy: 0.7783 - val_loss: 0.9137
Epoch 7/20
269/269 ━━━━━━━━━━━━━━━━━━━━ 21s 74ms/step - accuracy: 0.7162 - loss: 1.0060 - val_accuracy: 0.7363 - val_loss: 0.9585
Epoch 8/20
269/269 ━━━━━━━━━━━━━━━━━━━━ 29s 107ms/step - accuracy: 0.7630 - loss: 0.9772 

In [120]:
lstm.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (None, 50, 128)        │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,681,291 (44.56 MB)

 Trainable params: 3,893,763 (14.85 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 7,787,528 (29.71 MB)

In [121]:
loss, accuracy = lstm.evaluate(
    x_test,
    y_test
)
print("Loss:", loss)
print("Accuracy:", accuracy)

168/168 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.7661 - loss: 0.9144
Loss: 0.914402425289154
Accuracy: 0.7661320567131042


In [122]:
from sklearn.metrics import classification_report
import numpy as np

y_pred = lstm.predict(x_test)

y_pred = np.argmax(y_pred, axis=1)

print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Negative",
            "Neutral",
            "Positive"
        ]
    )
)

168/168 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step
              precision    recall  f1-score   support

    Negative       0.76      0.69      0.72      2071
     Neutral       0.00      0.00      0.00       272
    Positive       0.77      0.88      0.82      3019

    accuracy                           0.77      5362
   macro avg       0.51      0.53      0.52      5362
weighted avg       0.73      0.77      0.74      5362



c:\Users\mereh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\mereh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\mereh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave